In [1]:
# ============================================================
# CELL 1 - Environment Setup + Data Contract Checks
# ============================================================

import csv
import json
import os
import random
from datetime import datetime, timezone

import numpy as np
import torch
import torch.nn as nn
from google.colab import drive
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

drive.mount('/content/drive', force_remount=False)

# All files made by the new notebooks stay inside Outputs- New.
BASE = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs- New'
WESAD_DIR = os.path.join(BASE, 'WESAD')
AROAD_DIR = os.path.join(BASE, 'AffectiveROAD')

MASKED_DIR = os.path.join(BASE, 'MASKED_LSTM_AE')
CHECKPOINT_DIR = os.path.join(MASKED_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(MASKED_DIR, 'results')
MODELS_DIR = os.path.join(MASKED_DIR, 'models')
LOSO_MODELS_DIR = os.path.join(MODELS_DIR, 'loso')
FUSION_DIR = os.path.join(MASKED_DIR, 'fusion')

# The corrected five-minute sequences were already made by notebook 05.
SOURCE_CHECKPOINT_DIR = os.path.join(BASE, 'LSTM_AE', 'checkpoints')

for folder in [CHECKPOINT_DIR, RESULTS_DIR, MODELS_DIR, LOSO_MODELS_DIR, FUSION_DIR]:
    os.makedirs(folder, exist_ok=True)

FEATURE_NAMES = [
    'mean_HR', 'mean_RR', 'SDNN', 'RMSSD',
    'mean_BR', 'std_BR',
    'mean_temp', 'std_temp',
    'mean_acc_mag', 'std_acc_mag'
]
N_FEATURES = len(FEATURE_NAMES)
T = 5
SEED = 42
THRESHOLD_PERCENTILE = 95
MASK_RATIO = 0.20

# This exact string must also be returned by the deployed /predict endpoint.
MODEL_VERSION = 'c1-masked-lstm-ae-wesad-v2'

# S3 and S6 do not enter training, validation, or testing.
WESAD_SUBJECTS = [
    'S2', 'S4', 'S5', 'S7', 'S8', 'S9', 'S10',
    'S11', 'S13', 'S14', 'S15', 'S16', 'S17'
]
AROAD_DRIVES = [f'Drv{i}' for i in range(1, 14)]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

required_files = [
    *[
        os.path.join(SOURCE_CHECKPOINT_DIR, f'WESAD_{sid}_processed.npz')
        for sid in WESAD_SUBJECTS
    ],
    *[
        os.path.join(SOURCE_CHECKPOINT_DIR, f'AROAD_{drive_id}_processed.npz')
        for drive_id in AROAD_DRIVES
    ],
]
missing = [path for path in required_files if not os.path.exists(path)]
assert not missing, 'Run Cell 2 of the new unmasked notebook first. Missing:\n' + '\n'.join(missing)

print(f'Device: {device}')
print(f'Model version: {MODEL_VERSION}')
print(f'Training mask ratio: {MASK_RATIO:.0%}')
print(f'WESAD LOSO subjects: {len(WESAD_SUBJECTS)}')
print(f'AffectiveROAD external drives: {len(AROAD_DRIVES)}')
print('Plan: WESAD-only LOSO, then one final WESAD model, then AffectiveROAD external testing.')



Mounted at /content/drive
Device: cpu
Model version: c1-masked-lstm-ae-wesad-v2
Training mask ratio: 20%
WESAD LOSO subjects: 13
AffectiveROAD external drives: 13
Plan: WESAD-only LOSO, then one final WESAD model, then AffectiveROAD external testing.


In [2]:
# ============================================================
# CELL 2 - Load Corrected Sequences + Define Dynamic Masking
# ============================================================

def load_processed_subject(subject_id):
    path = os.path.join(SOURCE_CHECKPOINT_DIR, f'WESAD_{subject_id}_processed.npz')
    return np.load(path, allow_pickle=True)


def load_processed_drive(drive_id):
    path = os.path.join(SOURCE_CHECKPOINT_DIR, f'AROAD_{drive_id}_processed.npz')
    return np.load(path, allow_pickle=True)


def apply_mask(batch, mask_ratio=MASK_RATIO):
    """Randomly hide individual input values while keeping the clean target."""
    keep_mask = torch.rand_like(batch) > mask_ratio
    masked_batch = batch * keep_mask
    return masked_batch, keep_mask


sequence_summary = {'wesad': {}, 'affective_road': {}}
total_ar_sequences = 0

print('=' * 60)
print('Checking corrected sequences from notebook 05')
print('=' * 60)

for sid in WESAD_SUBJECTS:
    data = load_processed_subject(sid)
    base_seqs = data['base_seqs']
    stress_seqs = data['stress_seqs']
    assert base_seqs.ndim == 3 and base_seqs.shape[1:] == (T, N_FEATURES)
    assert stress_seqs.ndim == 3 and stress_seqs.shape[1:] == (T, N_FEATURES)
    assert np.isfinite(base_seqs).all() and np.isfinite(stress_seqs).all()
    sequence_summary['wesad'][sid] = {
        'baseline_sequences': int(len(base_seqs)),
        'stress_sequences': int(len(stress_seqs)),
    }
    print(f'{sid}: baseline={len(base_seqs)}, stress={len(stress_seqs)}')

for drive_id in AROAD_DRIVES:
    data = load_processed_drive(drive_id)
    base_seqs = data['base_seqs']
    stress_seqs = data['stress_seqs']
    assert base_seqs.ndim == 3 and base_seqs.shape[1:] == (T, N_FEATURES)
    assert stress_seqs.ndim == 3 and stress_seqs.shape[1:] == (T, N_FEATURES)
    assert np.isfinite(base_seqs).all() and np.isfinite(stress_seqs).all()
    sequence_summary['affective_road'][drive_id] = {
        'baseline_sequences': int(len(base_seqs)),
        'stress_sequences': int(len(stress_seqs)),
    }
    total_ar_sequences += len(base_seqs) + len(stress_seqs)

assert total_ar_sequences >= 30

with open(os.path.join(RESULTS_DIR, 'sequence_input_summary.json'), 'w') as file:
    json.dump(sequence_summary, file, indent=2)

# Verify that masking changes the input without changing its shape.
mask_test = torch.ones(32, T, N_FEATURES)
masked_test, keep_test = apply_mask(mask_test)
observed_mask_ratio = float((~keep_test).float().mean())
assert masked_test.shape == mask_test.shape
assert 0.10 <= observed_mask_ratio <= 0.30

print(f'\nAffectiveROAD held-out sequences: {total_ar_sequences}')
print(f'Masking utility check: {observed_mask_ratio:.1%} hidden in the test batch')



Checking corrected sequences from notebook 05
S2: baseline=15, stress=5
S4: baseline=15, stress=6
S5: baseline=15, stress=5
S7: baseline=15, stress=6
S8: baseline=15, stress=5
S9: baseline=15, stress=5
S10: baseline=15, stress=8
S11: baseline=15, stress=7
S13: baseline=14, stress=6
S14: baseline=15, stress=7
S15: baseline=15, stress=7
S16: baseline=15, stress=7
S17: baseline=15, stress=6

AffectiveROAD held-out sequences: 361
Masking utility check: 19.8% hidden in the test batch


In [3]:
# ============================================================
# CELL 3 - Masked LSTM Autoencoder + Evaluation Helpers
# ============================================================


class MaskedLSTMAutoEncoder(nn.Module):
    def __init__(self, n_features=N_FEATURES, hidden_size=64, n_layers=1):
        super().__init__()
        self.T = T
        self.encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
        )
        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
        )
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        _, (hidden, _) = self.encoder(x)
        bottleneck = hidden[-1]
        decoder_input = bottleneck.unsqueeze(1).repeat(1, self.T, 1)
        decoder_output, _ = self.decoder(decoder_input)
        return self.output_layer(decoder_output)


def train_masked_model(
    model,
    train_sequences,
    epochs=50,
    batch_size=32,
    lr=1e-3,
    mask_ratio=MASK_RATIO,
    seed=SEED,
    verbose=True,
):
    assert len(train_sequences) > 0, 'Training data is empty.'
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    dataset = TensorDataset(torch.tensor(train_sequences, dtype=torch.float32))
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, generator=generator)

    losses = []
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for (batch,) in loader:
            batch = batch.to(device)
            masked_batch, _ = apply_mask(batch, mask_ratio=mask_ratio)
            optimizer.zero_grad()
            reconstruction = model(masked_batch)
            # The target stays clean, matching the original masked notebook.
            loss = criterion(reconstruction, batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch)

        average_loss = total_loss / len(dataset)
        losses.append(average_loss)
        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(f'Epoch {epoch:3d}/{epochs}: masked loss={average_loss:.6f}')

    return model, losses


def score_sequences(model, sequences, batch_size=256):
    if len(sequences) == 0:
        return np.empty(0, dtype=float)

    model.eval()
    dataset = TensorDataset(torch.tensor(sequences, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    scores = []

    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(device)
            reconstruction = model(batch)
            errors = torch.mean((batch - reconstruction) ** 2, dim=(1, 2))
            scores.append(errors.cpu().numpy())

    return np.concatenate(scores).astype(float)


def compute_metrics(stress_scores, baseline_scores, threshold):
    y_true = np.concatenate([
        np.ones(len(stress_scores), dtype=int),
        np.zeros(len(baseline_scores), dtype=int),
    ])
    raw_scores = np.concatenate([stress_scores, baseline_scores])
    y_pred = (raw_scores > threshold).astype(int)

    return {
        'AUROC': float(roc_auc_score(y_true, raw_scores)),
        'F1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'FAR': float(np.mean(y_pred[y_true == 0])),
        'threshold': float(threshold),
        'n_baseline': int(len(baseline_scores)),
        'n_stress': int(len(stress_scores)),
    }


def to_predict_score(raw_error, baseline_p95):
    """Stable 0-1 score. The deployed /predict endpoint must use this exact formula."""
    raw_error = np.asarray(raw_error, dtype=float)
    return raw_error / (raw_error + float(baseline_p95) + 1e-12)


dummy = torch.zeros(4, T, N_FEATURES, dtype=torch.float32).to(device)
dummy_model = MaskedLSTMAutoEncoder().to(device)
assert dummy_model(dummy).shape == dummy.shape
del dummy_model, dummy
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Masked architecture and clean-evaluation scoring checks passed.')



Masked architecture and clean-evaluation scoring checks passed.


In [4]:
# ============================================================
# CELL 4 - Masked WESAD Leave-One-Subject-Out Evaluation
# ============================================================

loso_results = {}
out_of_fold_baseline_scores = []
out_of_fold_stress_scores = []

print('=' * 60)
print('MASKED WESAD LOSO CROSS-VALIDATION')
print('Each fold trains only on masked baseline data from the other 12 subjects.')
print('Held-out baseline and stress sequences are scored clean, with no masking.')
print('=' * 60)

for fold_index, test_sid in enumerate(WESAD_SUBJECTS):
    set_seed(SEED + fold_index)
    training_parts = []

    for train_sid in WESAD_SUBJECTS:
        if train_sid == test_sid:
            continue
        train_data = load_processed_subject(train_sid)
        training_parts.append(train_data['base_seqs'])

    X_train = np.concatenate(training_parts, axis=0).astype(np.float32)
    test_data = load_processed_subject(test_sid)
    X_test_baseline = test_data['base_seqs'].astype(np.float32)
    X_test_stress = test_data['stress_seqs'].astype(np.float32)

    model = MaskedLSTMAutoEncoder().to(device)
    model, losses = train_masked_model(
        model,
        X_train,
        epochs=50,
        batch_size=32,
        lr=1e-3,
        mask_ratio=MASK_RATIO,
        seed=SEED + fold_index,
        verbose=False,
    )

    baseline_scores = score_sequences(model, X_test_baseline)
    stress_scores = score_sequences(model, X_test_stress)
    threshold = float(np.percentile(baseline_scores, THRESHOLD_PERCENTILE))
    metrics = compute_metrics(stress_scores, baseline_scores, threshold)
    metrics['final_training_loss'] = float(losses[-1])
    loso_results[test_sid] = metrics

    out_of_fold_baseline_scores.append(baseline_scores)
    out_of_fold_stress_scores.append(stress_scores)

    torch.save(
        model.state_dict(),
        os.path.join(LOSO_MODELS_DIR, f'MASKED_LSTM_AE_LOSO_{test_sid}.pth'),
    )
    np.savez_compressed(
        os.path.join(RESULTS_DIR, f'MASKED_WESAD_LOSO_{test_sid}_scores.npz'),
        baseline_scores=baseline_scores,
        stress_scores=stress_scores,
    )
    print(f'{test_sid}: AUROC={metrics["AUROC"]:.4f}, F1={metrics["F1"]:.4f}')

loso_aurocs = [result['AUROC'] for result in loso_results.values()]
loso_f1s = [result['F1'] for result in loso_results.values()]
loso_summary = {
    'model_version': MODEL_VERSION,
    'protocol': 'WESAD leave-one-subject-out; baseline-only training with 20% dynamic masking',
    'mask_ratio': MASK_RATIO,
    'subjects': WESAD_SUBJECTS,
    'per_subject': loso_results,
    'mean_AUROC': float(np.mean(loso_aurocs)),
    'std_AUROC': float(np.std(loso_aurocs)),
    'mean_F1': float(np.mean(loso_f1s)),
    'std_F1': float(np.std(loso_f1s)),
}

with open(os.path.join(RESULTS_DIR, 'masked_wesad_loso_metrics.json'), 'w') as file:
    json.dump(loso_summary, file, indent=2)

np.save(
    os.path.join(RESULTS_DIR, 'masked_wesad_loso_heldout_baseline_raw_scores.npy'),
    np.concatenate(out_of_fold_baseline_scores),
)
np.save(
    os.path.join(RESULTS_DIR, 'masked_wesad_loso_heldout_stress_raw_scores.npy'),
    np.concatenate(out_of_fold_stress_scores),
)

print(f'\nMean LOSO AUROC: {loso_summary["mean_AUROC"]:.4f}')
print(f'Mean LOSO F1: {loso_summary["mean_F1"]:.4f}')



MASKED WESAD LOSO CROSS-VALIDATION
Each fold trains only on masked baseline data from the other 12 subjects.
Held-out baseline and stress sequences are scored clean, with no masking.
S2: AUROC=1.0000, F1=0.9091
S4: AUROC=1.0000, F1=0.9231
S5: AUROC=1.0000, F1=0.9091
S7: AUROC=1.0000, F1=0.9231
S8: AUROC=1.0000, F1=0.9091
S9: AUROC=1.0000, F1=0.9091
S10: AUROC=1.0000, F1=0.9412
S11: AUROC=1.0000, F1=0.9333
S13: AUROC=1.0000, F1=0.9231
S14: AUROC=1.0000, F1=0.9333
S15: AUROC=1.0000, F1=0.9333
S16: AUROC=1.0000, F1=0.9333
S17: AUROC=1.0000, F1=0.9231

Mean LOSO AUROC: 1.0000
Mean LOSO F1: 0.9233


In [5]:
# ============================================================
# CELL 5 - Train the Final Masked Model on All WESAD Baseline Data
# ============================================================

set_seed(SEED)
all_wesad_baseline = []
for sid in WESAD_SUBJECTS:
    subject_data = load_processed_subject(sid)
    all_wesad_baseline.append(subject_data['base_seqs'])

X_final_train = np.concatenate(all_wesad_baseline, axis=0).astype(np.float32)
print(f'Final training sequences: {X_final_train.shape}')

final_model = MaskedLSTMAutoEncoder().to(device)
final_model, final_losses = train_masked_model(
    final_model,
    X_final_train,
    epochs=50,
    batch_size=32,
    lr=1e-3,
    mask_ratio=MASK_RATIO,
    seed=SEED,
    verbose=True,
)

final_model_path = os.path.join(MODELS_DIR, 'MASKED_LSTM_AE_FINAL.pth')
torch.save(final_model.state_dict(), final_model_path)

# This threshold belongs to the deployed model and uses only its WESAD baseline training data.
# It is used to turn raw reconstruction error into the endpoint's stable 0-1 score.
final_training_raw_scores = score_sequences(final_model, X_final_train)
final_baseline_p95 = float(np.percentile(final_training_raw_scores, THRESHOLD_PERCENTILE))

np.save(
    os.path.join(RESULTS_DIR, 'masked_final_wesad_training_raw_scores.npy'),
    final_training_raw_scores,
)

model_metadata = {
    'model_version': MODEL_VERSION,
    'model_file': os.path.basename(final_model_path),
    'model_type': 'masked_lstm_autoencoder',
    'training_data': 'WESAD baseline only',
    'training_subjects': WESAD_SUBJECTS,
    'sequence_length_minutes': T,
    'n_features': N_FEATURES,
    'feature_names': FEATURE_NAMES,
    'hidden_size': 64,
    'n_layers': 1,
    'epochs': 50,
    'batch_size': 32,
    'learning_rate': 1e-3,
    'mask_ratio': MASK_RATIO,
    'training_target': 'original clean sequence',
    'evaluation_input': 'clean sequence with masking disabled',
    'seed': SEED,
    'baseline_threshold_percentile': THRESHOLD_PERCENTILE,
    'baseline_p95_raw_error': final_baseline_p95,
    'predict_score_formula': 'raw_error / (raw_error + baseline_p95_raw_error)',
    'predict_score_note': 'A monotonic anomaly score, not a probability.',
    'trained_at': datetime.now(timezone.utc).isoformat(),
}

with open(os.path.join(MODELS_DIR, 'MASKED_LSTM_AE_FINAL_metadata.json'), 'w') as file:
    json.dump(model_metadata, file, indent=2)

with open(os.path.join(RESULTS_DIR, 'masked_final_training_history.json'), 'w') as file:
    json.dump({'loss': [float(value) for value in final_losses]}, file, indent=2)

print(f'Final model saved: {final_model_path}')
print(f'Final WESAD baseline p95 raw error: {final_baseline_p95:.6f}')



Final training sequences: (194, 5, 10)
Epoch   1/50: masked loss=0.891530
Epoch  10/50: masked loss=0.743717
Epoch  20/50: masked loss=0.688917
Epoch  30/50: masked loss=0.626690
Epoch  40/50: masked loss=0.604534
Epoch  50/50: masked loss=0.585118
Final model saved: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs- New/MASKED_LSTM_AE/models/MASKED_LSTM_AE_FINAL.pth
Final WESAD baseline p95 raw error: 0.937913


In [6]:
# ============================================================
# CELL 6 - Masked AffectiveROAD External Test + Fusion Reference Vector
# ============================================================

external_results = {}
fusion_rows = []

print('=' * 60)
print('MASKED AFFECTIVEROAD EXTERNAL TEST')
print('This dataset was not used to train or choose the final model.')
print('=' * 60)

for drive_id in AROAD_DRIVES:
    data = load_processed_drive(drive_id)
    base_seqs = data['base_seqs'].astype(np.float32)
    stress_seqs = data['stress_seqs'].astype(np.float32)

    if len(base_seqs) == 0 or len(stress_seqs) == 0:
        print(f'{drive_id}: skipped because one class has no valid five-minute sequences')
        continue

    baseline_raw = score_sequences(final_model, base_seqs)
    stress_raw = score_sequences(final_model, stress_seqs)
    metrics = compute_metrics(stress_raw, baseline_raw, final_baseline_p95)
    external_results[drive_id] = metrics
    print(f'{drive_id}: AUROC={metrics["AUROC"]:.4f}, F1={metrics["F1"]:.4f}')

    base_output_scores = to_predict_score(baseline_raw, final_baseline_p95)
    stress_output_scores = to_predict_score(stress_raw, final_baseline_p95)

    for label_name, label_value, raw_values, output_values, starts, segments, subjective in [
        (
            'baseline', 0, baseline_raw, base_output_scores,
            data['base_start_sec'], data['base_segment_ids'], data['base_subjective_scores'],
        ),
        (
            'stress', 1, stress_raw, stress_output_scores,
            data['stress_start_sec'], data['stress_segment_ids'], data['stress_subjective_scores'],
        ),
    ]:
        for raw_error, output_score, start_sec, segment_id, subjective_score in zip(
            raw_values, output_values, starts, segments, subjective
        ):
            fusion_rows.append({
                'c1_score': float(output_score),
                'raw_reconstruction_error': float(raw_error),
                'evaluation_label': label_value,
                'evaluation_class': label_name,
                'drive_id': drive_id,
                'segment_id': str(segment_id),
                'sequence_start_sec': float(start_sec),
                'subjective_stress_score': (
                    float(subjective_score) if np.isfinite(subjective_score) else ''
                ),
                'model_version': MODEL_VERSION,
            })

assert len(fusion_rows) >= 30, 'Fusion reference requires at least 30 held-out scores.'
fusion_rows.sort(key=lambda row: (int(row['drive_id'].replace('Drv', '')), row['sequence_start_sec']))

heldout_scores = np.asarray([row['c1_score'] for row in fusion_rows], dtype=np.float32)
assert np.isfinite(heldout_scores).all()
assert np.all((heldout_scores >= 0.0) & (heldout_scores <= 1.0))

npy_path = os.path.join(FUSION_DIR, 'c1_masked_heldout_scores.npy')
csv_path = os.path.join(FUSION_DIR, 'c1_masked_heldout_scores.csv')
json_path = os.path.join(FUSION_DIR, 'c1_masked_heldout_scores.json')

np.save(npy_path, heldout_scores)

with open(csv_path, 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=list(fusion_rows[0].keys()))
    writer.writeheader()
    writer.writerows(fusion_rows)

fusion_payload = {
    'model_version': MODEL_VERSION,
    'source': 'AffectiveROAD held-out external evaluation',
    'score_field': 'c1_score',
    'score_formula': model_metadata['predict_score_formula'],
    'n_scores': int(len(heldout_scores)),
    'scores': [float(score) for score in heldout_scores],
}
with open(json_path, 'w') as file:
    json.dump(fusion_payload, file, indent=2)

valid_results = list(external_results.values())
external_summary = {
    'model_version': MODEL_VERSION,
    'protocol': 'Final WESAD-only model tested on untouched AffectiveROAD sequences',
    'fixed_wesad_threshold': final_baseline_p95,
    'per_drive': external_results,
    'macro_mean_AUROC': float(np.mean([result['AUROC'] for result in valid_results])),
    'macro_mean_F1': float(np.mean([result['F1'] for result in valid_results])),
    'fusion_reference_count': int(len(heldout_scores)),
    'fusion_reference_files': [npy_path, csv_path, json_path],
}

with open(os.path.join(RESULTS_DIR, 'masked_affectiveroad_external_metrics.json'), 'w') as file:
    json.dump(external_summary, file, indent=2)

print('\nExternal test complete.')
print(f'Macro mean AUROC: {external_summary["macro_mean_AUROC"]:.4f}')
print(f'Macro mean F1: {external_summary["macro_mean_F1"]:.4f}')
print(f'Fusion held-out score count: {len(heldout_scores)}')
print(f'Model version: {MODEL_VERSION}')
print(f'Fusion files saved in: {FUSION_DIR}')



MASKED AFFECTIVEROAD EXTERNAL TEST
This dataset was not used to train or choose the final model.
Drv1: AUROC=1.0000, F1=1.0000
Drv2: AUROC=1.0000, F1=0.8571
Drv3: AUROC=0.9610, F1=0.0000
Drv4: AUROC=0.6042, F1=0.0000
Drv5: AUROC=0.9371, F1=0.7500
Drv6: AUROC=1.0000, F1=1.0000
Drv7: AUROC=0.5688, F1=0.0000
Drv8: AUROC=0.7348, F1=0.1538
Drv9: AUROC=1.0000, F1=0.9231
Drv10: AUROC=1.0000, F1=1.0000
Drv11: AUROC=0.8611, F1=0.0000
Drv12: AUROC=1.0000, F1=0.8571
Drv13: AUROC=0.6176, F1=0.0000

External test complete.
Macro mean AUROC: 0.8680
Macro mean F1: 0.5032
Fusion held-out score count: 361
Model version: c1-masked-lstm-ae-wesad-v2
Fusion files saved in: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs- New/MASKED_LSTM_AE/fusion


In [7]:
# ============================================================
# CELL 7 - Pooled and Weighted Masked AffectiveROAD Metrics
# ============================================================

from sklearn.metrics import confusion_matrix

y_true = np.asarray([row['evaluation_label'] for row in fusion_rows], dtype=int)
pooled_raw_scores = np.asarray(
    [row['raw_reconstruction_error'] for row in fusion_rows], dtype=float
)
y_pred = (pooled_raw_scores > final_baseline_p95).astype(int)

pooled_auroc = float(roc_auc_score(y_true, pooled_raw_scores))
pooled_f1 = float(f1_score(y_true, y_pred, zero_division=0))
pooled_precision = float(precision_score(y_true, y_pred, zero_division=0))
pooled_recall = float(recall_score(y_true, y_pred, zero_division=0))
pooled_accuracy = float(np.mean(y_pred == y_true))

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
pooled_far = float(fp / (fp + tn)) if (fp + tn) else 0.0

drive_weights = np.asarray(
    [result['n_baseline'] + result['n_stress'] for result in valid_results],
    dtype=float,
)
weighted_auroc = float(np.average(
    [result['AUROC'] for result in valid_results], weights=drive_weights
))
weighted_f1 = float(np.average(
    [result['F1'] for result in valid_results], weights=drive_weights
))

pooled_results = {
    'model_version': MODEL_VERSION,
    'n_total_sequences': int(len(y_true)),
    'n_baseline_sequences': int(np.sum(y_true == 0)),
    'n_stress_sequences': int(np.sum(y_true == 1)),
    'pooled_metrics': {
        'AUROC': pooled_auroc,
        'F1': pooled_f1,
        'precision': pooled_precision,
        'recall': pooled_recall,
        'accuracy': pooled_accuracy,
        'FAR': pooled_far,
    },
    'confusion_matrix': {
        'true_negative': int(tn),
        'false_positive': int(fp),
        'false_negative': int(fn),
        'true_positive': int(tp),
    },
    'drive_size_weighted_metrics': {
        'AUROC': weighted_auroc,
        'F1': weighted_f1,
    },
}

pooled_path = os.path.join(RESULTS_DIR, 'masked_affectiveroad_pooled_metrics.json')
with open(pooled_path, 'w') as file:
    json.dump(pooled_results, file, indent=2)

print('=' * 60)
print('MASKED AFFECTIVEROAD COMPLETE EXTERNAL RESULTS')
print('=' * 60)
print(f'Pooled AUROC    : {pooled_auroc:.4f}')
print(f'Pooled F1       : {pooled_f1:.4f}')
print(f'Pooled precision: {pooled_precision:.4f}')
print(f'Pooled recall   : {pooled_recall:.4f}')
print(f'Pooled accuracy : {pooled_accuracy:.4f}')
print(f'Pooled FAR      : {pooled_far:.4f}')
print(f'Confusion matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}')
print(f'Weighted AUROC  : {weighted_auroc:.4f}')
print(f'Weighted F1     : {weighted_f1:.4f}')
print(f'Saved to: {pooled_path}')


MASKED AFFECTIVEROAD COMPLETE EXTERNAL RESULTS
Pooled AUROC    : 0.8562
Pooled F1       : 0.5419
Pooled precision: 0.9545
Pooled recall   : 0.3784
Pooled accuracy : 0.8033
Pooled FAR      : 0.0080
Confusion matrix: TN=248, FP=2, FN=69, TP=42
Weighted AUROC  : 0.8628
Weighted F1     : 0.4732
Saved to: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs- New/MASKED_LSTM_AE/results/masked_affectiveroad_pooled_metrics.json
